In [0]:
from pyspark.sql import functions as F

login = "mariiasirska"
catalog = "dbr_dev_ua5816bd"
bronze = f"{catalog}.{login}_bronze"
silver = f"{catalog}.{login}_silver"
gold = f"{catalog}.{login}_gold"

events_src = f"{bronze}.lab3_events_bronze"

In [0]:
for t in [f"{silver}.lp_orders_bronze", f"{silver}.lp_orders_silver", f"{silver}.lp_events_silver",
          f"{gold}.lp_orders_by_status", f"{gold}.lp_events_by_device"]:
    print(t, spark.table(t).count())

display(spark.table(f"{gold}.lp_orders_by_status"))

In [0]:
base = spark.table(events_src).limit(5)
bad = (
    base.withColumn("event_id", F.lit(None).cast("string"))
    .unionByName(base.withColumn("temperature", F.lit(999.0)))
    .unionByName(base.withColumn("humidity", F.lit(150.0)))
)
bad.write.mode("append").saveAsTable(events_src)
print("appended bad events:", bad.count())


 Expectation metrics from the pipeline event log:

In [0]:
def expectation_metrics(table_name):
    ev = spark.sql(f"SELECT * FROM event_log(TABLE({table_name}))")
    return (
        ev.filter("event_type = 'flow_progress'")
        .selectExpr(
            "timestamp",
            "origin.flow_name as flow",
            "explode(from_json(details:flow_progress.data_quality.expectations, "
            "'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) as e",
        )
        .select("timestamp", "flow", "e.name", "e.passed_records", "e.failed_records")
        .orderBy(F.desc("timestamp"))
    )

for t in [f"{silver}.lp_events_silver", f"{silver}.lp_orders_silver"]:
    try:
        display(expectation_metrics(t))
    except Exception as e:
        print(t, "-> event log not available:", str(e)[:200])
